# Natural frequencies

The lowest natural frequencies of a cylinder are clustered, and the tests
compute them with a deterministic and verified eigenvalue solver, which
checks the residual of each eigenpair and, with Sylvester's law of inertia,
that no eigenvalue is missing, the function `lowest_frequencies` of
`tests/natural_frequency_solver.py`:

**Theory of the solver.** The natural frequencies solve the generalized
symmetric eigenvalue problem

$$
\left( \boldsymbol{K} - \omega_n^2 \boldsymbol{M} \right)
\boldsymbol{\phi} = \boldsymbol{0}
$$

with $\boldsymbol{K}$ positive definite and $\boldsymbol{M}$ the consistent
mass matrix, see [Analyses](../analyses.rst). The frequencies of a closed
cylinder come in pairs of equal values, the modes $\cos(n\theta)$ and
$\sin(n\theta)$ of the same number $n$ of circumferential waves, and the
lowest pairs are close to each other. `lowest_frequencies` uses the shift
$-1$, below the lowest eigenvalue: the largest eigenvalues $\mu$ of
$(\boldsymbol{K} + \boldsymbol{M})^{-1} \boldsymbol{M}$ give $\omega_n^2 =
1/\mu - 1$, computed by ARPACK from a fixed starting vector. The result is
verified in two ways:

- the relative residual $\| \boldsymbol{K} \boldsymbol{\phi} - \omega_n^2
  \boldsymbol{M} \boldsymbol{\phi} \| / \| \boldsymbol{K} \boldsymbol{\phi}
  \|$ of each eigenpair is below `rtol`;
- by Sylvester's law of inertia, the number of negative pivots of the
  symmetric factorization $\boldsymbol{L} \boldsymbol{D}
  \boldsymbol{L}^\top$ of $\boldsymbol{K} - s \boldsymbol{M}$ is the number
  of eigenvalues below $s$. With $s$ slightly above the highest eigenvalue
  returned, it must equal the number of eigenvalues returned, such that
  none is missing.

In [1]:
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import LinearOperator, eigs, splu

In [2]:
def negative_pivots(A):
    r"""Number of negative eigenvalues of the symmetric sparse matrix ``A``,
    from an `\boldsymbol{L}\boldsymbol{D}\boldsymbol{L}^\top` factorization with a symmetric permutation"""
    lu = splu(csc_matrix(A), permc_spec='MMD_AT_PLUS_A',
              diag_pivot_thresh=0., options=dict(SymmetricMode=True))
    assert np.array_equal(lu.perm_r, lu.perm_c), 'row interchange'
    d = lu.U.diagonal()
    assert np.all(np.isfinite(d)) and np.all(d != 0)
    return int(np.count_nonzero(d < 0))

In [3]:
def lowest_frequencies(K, M, k, rtol=1e-8):
    r"""The ``k`` lowest `\omega_n^2` and modes of `\boldsymbol{K} \boldsymbol{u} = \omega_n^2
    \boldsymbol{M} \boldsymbol{u}`, ``K`` positive definite, verified

    Notes
    -----
    Shift-invert with the shift `-1`, below the lowest eigenvalue: the
    largest eigenvalues `\mu` of `(\boldsymbol{K} + \boldsymbol{M})^{-1}
    \boldsymbol{M}` give `\omega_n^2 = 1/\mu - 1`.

    """
    # shift-invert with the shift -1, below the lowest eigenvalue
    lu = splu(csc_matrix(K + M))
    op = LinearOperator(K.shape, matvec=lambda x: lu.solve(M @ x),
                        dtype=float)
    mu, eigvecs = eigs(op, k=k, which='LM', tol=0, ncv=4*k,
                       v0=np.ones(K.shape[0]))
    assert np.all(np.abs(mu.imag) <= 1e-10*np.abs(mu))
    eigvals = 1/mu.real - 1
    # real modes: remove the arbitrary complex phase of each vector
    i = np.argmax(np.abs(eigvecs), axis=0)
    phase = eigvecs[i, np.arange(k)]
    eigvecs = (eigvecs*(np.abs(phase)/phase)).real
    order = np.argsort(eigvals)
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    for lam, x in zip(eigvals, eigvecs.T):
        Kx = K @ x
        res = np.linalg.norm(Kx - lam*(M @ x))/np.linalg.norm(Kx)
        assert res < rtol, 'eigenpair residual %.1e' % res
    # the next eigenvalue, if any, is above the last one returned; the
    # shift lies in the gap between them, away from a multiple eigenvalue
    s = eigvals[-1]*(1 + 1e-6)
    assert negative_pivots(K - s*M) == k, 'eigenvalues missing'
    return eigvals, eigvecs

A closed isotropic cylinder with $u$, $v$ and $w$ constrained at the nodes
of both ends. The mass matrix of `update_M` requires the thickness `h` and
the density `rho` of each element. The code is that of the test
`tests/test_natural_frequency.py`:

**What the test verifies.** An isotropic cylinder, $R = 0.4$ m, $L =
0.8$ m, $h = 1$ mm, $E = 70$ GPa, $\nu = 0.33$ and $\rho = 2700$ kg/m$^3$,
with $u$, $v$ and $w$ constrained at the nodes of both ends. The 16 lowest
eigenvalues are computed with `lowest_frequencies`, and the lowest angular
frequency $\omega_1$ is checked within 1 % against 1096.108 rad/s, a result
of the element stored in the test (a regression value, not a published
one). The mass matrix includes the rotary inertia of the rotations of the
normal of the kinematics, see
[Element matrices and internal forces](../matrices.rst). The test
uses the Donnell-type kinematics, which lose accuracy for modes with few
circumferential waves [Leissa1973]; for the lowest modes the Sanders-type
kinematics are the safer choice, see [Kinematics of the cylindrical shell](../kinematics.rst).

In [4]:
from numpy import isclose, pi
from scipy.sparse import coo_matrix
from composites import isotropic_plate

from bfsccylinder import (BFSCCylinder, update_KC0, update_M, DOF, DOUBLE, INT,
KC0_SPARSE_SIZE, M_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

In [5]:
def test_natural_frequency(plot=False):
    # number of nodes
    nx = 21 # axial, keep odd if you want a line of nodes exactly in the middle
    ny = 56 # circumferential

    # geometry
    L = 0.8 # m
    R = 0.4 # m
    b = 2*pi*R # m

    # material properties
    E = 70e9 # Pa
    nu = 0.33
    h = 0.001 # m
    rho = 2.7e3 # kg/m3
    prop = isotropic_plate(thickness=h, E=E, nu=nu)

    nids = 1 + np.arange(nx*(ny+1))
    nids_mesh = nids.reshape(nx, ny+1)
    # closing the cylinder by reassigning last row of node-ids
    nids_mesh[:, -1] = nids_mesh[:, 0]
    nids = np.unique(nids_mesh)
    nid_pos = dict(zip(nids, np.arange(len(nids))))

    xlin = np.linspace(0, L, nx)
    ytmp = np.linspace(0, b, ny+1)
    ylin = np.linspace(0, b-(ytmp[-1] - ytmp[-2]), ny)
    xmesh, ymesh = np.meshgrid(xlin, ylin)
    xmesh = xmesh.T
    ymesh = ymesh.T

    # getting nodes
    ncoords = np.vstack((xmesh.flatten(), ymesh.flatten())).T
    x = ncoords[:, 0]
    y = ncoords[:, 1]

    n1s = nids_mesh[:-1, :-1].flatten()
    n2s = nids_mesh[1:, :-1].flatten()
    n3s = nids_mesh[1:, 1:].flatten()
    n4s = nids_mesh[:-1, 1:].flatten()

    nint = 4
    points, weights = get_points_weights(nint=nint)

    num_elements = len(n1s)
    print('num_elements', num_elements)

    elements = []
    N = DOF*nx*ny
    Kr = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    Kc = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    Kv = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    Mr = np.zeros(M_SPARSE_SIZE*num_elements, dtype=INT)
    Mc = np.zeros(M_SPARSE_SIZE*num_elements, dtype=INT)
    Mv = np.zeros(M_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    init_k_KC0 = 0
    init_k_M = 0
    for n1, n2, n3, n4 in zip(n1s, n2s, n3s, n4s):
        shell = BFSCCylinder(nint)
        shell.c1 = DOF*nid_pos[n1]
        shell.c2 = DOF*nid_pos[n2]
        shell.c3 = DOF*nid_pos[n3]
        shell.c4 = DOF*nid_pos[n4]
        shell.R = R
        shell.h = h
        shell.rho = rho
        shell.lex = L/(nx-1)
        shell.ley = b/ny
        assign_constant_ABD(shell, prop)
        shell.init_k_KC0 = init_k_KC0
        shell.init_k_M = init_k_M
        update_KC0(shell, points, weights, Kr, Kc, Kv)
        update_M(shell, Mr, Mc, Mv)
        init_k_KC0 += KC0_SPARSE_SIZE
        init_k_M += M_SPARSE_SIZE
        elements.append(shell)

    KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()
    M = coo_matrix((Mv, (Mr, Mc)), shape=(N, N)).tocsc()

    print('structural matrices OK')

    # applying boundary conditions
    bk = np.zeros(N, dtype=bool)

    # simply supports
    checkSS = isclose(x, 0) | isclose(x, L)
    bk[0::DOF] = checkSS
    bk[3::DOF] = checkSS
    bk[6::DOF] = checkSS

    bu = ~bk # same as np.logical_not, defining unknown DOFs

    # sub-matrices corresponding to unknown DOFs
    Kuu = KC0[bu, :][:, bu]
    Muu = M[bu, :][:, bu]

    # solving for natural frequencies
    num_eigenvalues = 16
    # deterministic and verified, see natural_frequency_solver.py
    eigvals, eigvecsu = lowest_frequencies(Kuu, Muu, num_eigenvalues)
    omegan = eigvals**0.5

    mode = 0
    uu = eigvecsu[:, mode]
    u = np.zeros(N, dtype=float)
    u[bu] = uu

    w = u[6::DOF].reshape(nx, ny)
    print('omegan', omegan)

    assert isclose(omegan[0], 1096.10818933, rtol=0.01)

    if plot:
        import matplotlib
        matplotlib.use('TkAgg')
        import matplotlib.pyplot as plt
        from matplotlib import cm
        plt.gca().set_aspect('equal')
        levels = np.linspace(w.min(), w.max(), 300)
        plt.contourf(xmesh, ymesh, w, levels=levels, cmap=cm.jet)
        plt.colorbar()
        plt.show()

In [6]:
test_natural_frequency()

num_elements 1120


structural matrices OK


omegan [1096.07315122 1096.07315122 1104.67747867 1104.67747867 1172.82680411
 1172.82680411 1219.38492714 1219.38492714 1313.43725165 1313.43725166
 1459.0792155  1459.0792155  1500.9214068  1500.9214068  1724.31301527
 1724.31301527]


## References

- **[Leissa1973]** Leissa, A. W., Vibration of Shells, NASA SP-288, National Aeronautics and
  Space Administration, Washington, DC, 1973.